# A2 — Reconhecimento semântico em publicidade visual com CLIP (ADS-16)

**Como executar:** Google Colab com GPU T4 (*Ambiente de execução > Alterar o tipo de ambiente de
execução > T4 GPU*) e *Executar tudo*. Nenhum modelo é treinado: o notebook usa só os embeddings de
um CLIP pré-treinado.

| | Execução completa |
|---|---|
| Tempo estimado de execução | [preencher após a execução final] |
| Pico de VRAM alocada (reservada) | [preencher] |
| Pico de RAM do processo | [preencher] |

> [Data e forma da medição, como na A1.]

## Setup

Clone do código (`src/`), pasta de saída, imports e configuração. O código dos módulos está no
repositório; o notebook só orquestra.

In [ ]:
import os
import subprocess
import time

_T_INICIO_NOTEBOOK = time.perf_counter()  # tempo total do notebook (R15)
TEMPOS = []  # registro das etapas cronometradas

REPO_URL = "https://github.com/anacaetano02/Deep-Learning-Visao-Computacional.git"
SUBPASTA = "A2_clip_ads16"
REPO_DIR = "/content/repo"
# Versão do código: na entrega, a tag da A2 (ex.: "a2-entrega"). None = branch principal.
GIT_REF = None

PROJECT_DIR = os.path.join(REPO_DIR, SUBPASTA)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # --filter=blob:none + --sparse: baixa só os arquivos da pasta da A2, não o repositório inteiro
    ref = ["--branch", GIT_REF] if GIT_REF else []
    subprocess.run(["git", "clone", "--filter=blob:none", "--sparse", "--depth", "1", *ref, REPO_URL, REPO_DIR],
                   check=True)
    subprocess.run(["git", "sparse-checkout", "set", SUBPASTA], cwd=REPO_DIR, check=True)
elif GIT_REF:
    # Já clonado nesta sessão: garante a versão fixada (não puxa commits novos)
    subprocess.run(["git", "fetch", "--depth", "1", "origin", "tag", GIT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", GIT_REF], cwd=REPO_DIR, check=True)
else:
    print(f"Repositório já clonado em '{REPO_DIR}': buscando mudanças mais recentes...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

TEMPOS.append({"etapa": "clone/pull do repositório", "segundos": round(time.perf_counter() - _T_INICIO_NOTEBOOK, 1)})

In [ ]:
import importlib, sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Ao reexecutar o notebook na mesma sessão depois de atualizar o código, recarrega os módulos src.*
# já importados. Num runtime novo não há nada para recarregar e a célula não faz nada.
for nome in sorted(m for m in sys.modules if m == "src" or m.startswith("src.")):
    importlib.reload(sys.modules[nome])

In [ ]:
# Onde salvar os artefatos desta execução (embeddings, tabelas, figuras).
# * False (padrão): disco local do Colab, em /content (apagado quando o runtime é desconectado).
# * True: Google Drive (pede autorização no início).
from pathlib import Path

SALVAR_NO_DRIVE = False

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/deep_learning_visao_computacional/A2_clip_ads16")
else:
    BASE_DIR = Path("/content/A2_clip_ads16_saida")

DIR_DADOS = BASE_DIR / "data"
DIR_OUTPUTS = BASE_DIR / "outputs"
DIR_REPORT_ASSETS = DIR_OUTPUTS / "report_assets"   # tabelas e figuras que vão para o relatório

for d in (DIR_DADOS, DIR_REPORT_ASSETS):
    d.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR = {BASE_DIR} (Drive: {SALVAR_NO_DRIVE})")

In [ ]:
# Configuração do experimento (decisões do aluno; justificar em markdown)
SEED = 42
CHECKPOINT_CLIP = None   # [definir: ex. id do Hugging Face]
REVISAO_CLIP = None      # [fixar por SHA, como na A1]

## Dados: ADS-16 (R1, R2)

[Origem e download do dataset; quantos anúncios e categorias existem de fato.]

In [ ]:
import kagglehub

from src.dados import ASSINATURA_DATASET, N_ARQUIVOS_DATASET, impressao_digital

# Versão do dataset no Kaggle. No Colab, o kagglehub pode servir o cache do próprio Colab
# (/kaggle/input/...), cujo caminho não mostra a versão: quem garante que os dados são os mesmos
# é a impressão digital (nº de arquivos + hash da lista caminho|tamanho).
VERSAO_ADS16 = 1

_t0 = time.perf_counter()
DIR_ADS16 = Path(kagglehub.dataset_download(f"groffo/ads16-dataset/versions/{VERSAO_ADS16}"))
TEMPOS.append({"etapa": "download do ADS-16", "segundos": round(time.perf_counter() - _t0, 1)})
print(DIR_ADS16)

n_arquivos, mb, assinatura = impressao_digital(DIR_ADS16)
assert (n_arquivos, assinatura) == (N_ARQUIVOS_DATASET, ASSINATURA_DATASET), "O conteúdo do ADS-16 mudou"
print(f"{n_arquivos} arquivos, {mb} MB, assinatura {assinatura}")

In [ ]:
from collections import Counter

from src.dados import arvore

arvore(DIR_ADS16)

In [ ]:
# Extensões no dataset inteiro. Os 6 .zip são os originais (Ads/Corpus/Documents de cada parte),
# já extraídos nas pastas ao lado: não escondem anúncios. O único .jpg está em Corpus/ (foto pessoal).
extensoes = Counter(p.suffix.lower() for p in DIR_ADS16.rglob("*") if p.is_file())
print(extensoes.most_common())
print(sorted(str(p.relative_to(DIR_ADS16)) for p in DIR_ADS16.rglob("*.zip")))

In [ ]:
from PIL import Image

from src.dados import N_ARQUIVOS_ADS, N_CATEGORIAS, carregar_imagem, listar_anuncios

# Corpus = só os anúncios (Ads/); Corpus/ (fotos pessoais) e Documents/ ficam de fora.
anuncios = listar_anuncios(DIR_ADS16)
por_categoria = anuncios.groupby("categoria").size()
assert len(anuncios) == N_ARQUIVOS_ADS, len(anuncios)
assert sorted(por_categoria.index) == list(range(1, N_CATEGORIAS + 1)), por_categoria
print(f"{len(anuncios)} arquivos de anúncio em {por_categoria.size} categorias")
print("Categorias fora do padrão de 15:", por_categoria[por_categoria != 15].to_dict())

# Duplicatas exatas (mesmo conteúdo em bytes): mantém a primeira ocorrência
duplicatas = anuncios[anuncios.duplicated("md5", keep=False)]
print(f"Duplicatas exatas: {len(duplicatas)} arquivos")
if len(duplicatas):
    display(duplicatas[["caminho_rel", "categoria", "n", "largura", "altura"]])

corpus = anuncios.drop_duplicates("md5", keep="first").reset_index(drop=True)
corpus["caminho"] = [str(DIR_ADS16 / c) for c in corpus["caminho_rel"]]
corpus.drop(columns="caminho").to_csv(DIR_REPORT_ASSETS / "corpus_ads16.csv", index=False)
print(f"Corpus final: {len(corpus)} anúncios")

In [ ]:
import matplotlib.pyplot as plt

# EDA do corpus, focada no que afeta o CLIP:
# * modo de cor / transparência: tratados em carregar_imagem (fundo branco);
# * proporção: o processor do CLIP redimensiona o menor lado para 224 px e faz center crop
#   quadrado, então o lado maior perde (1 - menor/maior) da sua extensão, metade de cada lado;
# * resolução: anúncios de ~500 px viram 224 px, e texto pequeno tende a ficar ilegível.
print("Modos de cor:", Counter(corpus["modo"]).most_common())
print(f"Com transparência (compostos sobre fundo branco): {int(corpus['transparente'].sum())} anúncios")
display(corpus[["largura", "altura", "proporcao"]].describe().round(2))

print("Tamanhos distintos (largura×altura):")
display((corpus["largura"].astype(str) + "×" + corpus["altura"].astype(str)).value_counts().to_frame("anúncios"))

menor = corpus[["largura", "altura"]].min(axis=1)
maior = corpus[["largura", "altura"]].max(axis=1)
corpus["perda_crop_por_lado"] = (1 - menor / maior) / 2
nao_quadrados = corpus[corpus["largura"] != corpus["altura"]]
print(f"Não quadrados: {len(nao_quadrados)} de {len(corpus)}; perda máxima no center crop: "
      f"{corpus['perda_crop_por_lado'].max():.1%} de cada lado do lado maior")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(corpus["proporcao"], bins=30)
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_xlabel("largura / altura (1 = quadrado)")
ax.set_ylabel("nº de anúncios")
ax.set_title("Proporção dos anúncios do ADS-16")
plt.tight_layout()
fig.savefig(DIR_REPORT_ASSETS / "proporcao_anuncios.png", dpi=150)
plt.show()

# Anúncios com transparência: convert("RGB") direto × composição sobre fundo branco
transp = corpus[corpus["transparente"]]
if len(transp):
    fig, eixos = plt.subplots(2, len(transp), figsize=(2.6 * len(transp), 5.5), squeeze=False)
    for k, (_, r) in enumerate(transp.iterrows()):
        with Image.open(r["caminho"]) as img:
            eixos[0, k].imshow(img.convert("RGB"))
        eixos[1, k].imshow(carregar_imagem(r["caminho"]))
        eixos[0, k].set_title(f"cat. {r['categoria']} · {r['n']}.png ({r['modo']})", fontsize=8)
    eixos[0, 0].set_ylabel('convert("RGB")')
    eixos[1, 0].set_ylabel("fundo branco")
    for ax in eixos.flat:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Um anúncio sorteado por categoria (diversidade do corpus, R2)
exemplos = corpus.groupby("categoria").sample(n=1, random_state=SEED)
fig, eixos = plt.subplots(4, 5, figsize=(16, 13))
for ax, (_, r) in zip(eixos.flat, exemplos.iterrows()):
    ax.imshow(carregar_imagem(r["caminho"]))
    ax.set_title(f"categoria {r['categoria']} · {r['n']}.png\n{r['largura']}×{r['altura']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
fig.savefig(DIR_REPORT_ASSETS / "exemplos_por_categoria.png", dpi=120)
plt.show()

Com 300 anúncios, usar o corpus inteiro é a unica opçao já que o subconjunto não existe. 

* o ADS-16 tem 300 anúncios e que todos foram usados;
* o mínimo de 500 vale para subconjuntos, e por isso não se aplica;

Fotos pessoais têm outra distribuição: não têm logo, texto de marketing nem produto em destaque. Misturá-las inflaria o corpus com imagens que mudam o ranking. Por exemplo, "a smiling person" subiria artificialmente.


## Modelo CLIP e embeddings

[Checkpoint escolhido e por quê. Nenhum treino: só extração de embeddings.]

In [ ]:
# [carregar o CLIP (revisão fixada) e o processor]

In [ ]:
# [embeddings normalizados de todas as imagens do corpus]

## 2.1 — Ranking de objetos por frequência semântica

### Descrições de objetos/conceitos (R3)

[As ≥ 20 descrições e o critério de escolha.]

In [ ]:
# [lista de descrições e seus embeddings de texto]

In [ ]:
# [matriz de cosine similarity imagem × descrição]

### Threshold de similaridade (R4, R5)

[Valor escolhido e justificativa, com base na distribuição das similaridades.]

In [ ]:
# [distribuição das similaridades e definição do threshold]

### Ranking (R6)

In [ ]:
# [ranking: score médio de similaridade e frequência acima do threshold]

### Os 5 objetos mais encontrados (R7)

In [ ]:
# [visualização dos 5 mais frequentes com exemplos do corpus]

**Interpretação do ranking.** [O que o ranking mostra sobre o corpus.]

## 2.2 — Busca semântica por consulta textual

### Função de busca (R8)

In [ ]:
# [busca: consulta -> top-5 imagens mais similares]

### Consultas (R9)

[As ≥ 8 consultas, organizadas de genérico → específico e de concreto → abstrato.]

In [ ]:
# [executar e exibir o top-5 de cada consulta (R10)]

### Análise das consultas (R11)

[Para cada consulta: recuperou o que descreve ou interpretou de forma inesperada? Por quê?]

## Discussão (R12, R13, R14)

### Alinhamento das representações visuais e textuais (R12)

[texto]

### Por que o pré-treino contrastivo permite recuperação sem treino supervisionado (R13)

[texto]

### Consulta textual do CLIP × tokenização no BERT: padding e attention mask (R14)

[texto]

## Apêndice: versões e tempo/memória (R15)

In [ ]:
# [versões das bibliotecas e resumo de tempo e memória, como na A1]